# Training Pipeline
---
### Baseline · Optuna Tuning · Evaluation · Model Saving

## Configuration

### Imports

In [1]:
import warnings

import joblib
import numpy as np
import optuna
from optuna.samplers import TPESampler
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

/Users/kzas/Documents/Vertex AI Instance/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Constants

In [2]:
# Paths
FOLDER = "data/sen/"
TRAIN_PATH = FOLDER + "train.csv"
TEST_PATH = FOLDER + "test.csv"
SELECTED_FEATURES_PATH = FOLDER + "selected_features.txt"
RESULTS_PATH = FOLDER + "results.txt"
MODEL_DIR = "models/"

# Parameters
TARGET = "agb"
with open(SELECTED_FEATURES_PATH, 'r', encoding="utf-8") as f:
    FEATURES = [line.strip() for line in f if line.strip()]
SEED = 42
N_SPLITS = 5
N_TRIALS = 50

SAVE_DATE = pd.Timestamp.now().strftime("%Y%m%d")

# KFold
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

print("Configuration loaded.")


Configuration loaded.


## Data Loading

### Load train & test sets

In [3]:
train_data = pd.read_csv(TRAIN_PATH, usecols=FEATURES + [TARGET])
test_data = pd.read_csv(TEST_PATH,  usecols=FEATURES + [TARGET])

X_train = train_data[FEATURES].copy()
y_train = train_data[TARGET].copy()

X_test = test_data[FEATURES].copy()
y_test = test_data[TARGET].copy()

print(f"Train : {X_train.shape[0]} obs × {X_train.shape[1]} features")
print(f"Test  : {X_test.shape[0]} obs × {X_test.shape[1]} features")


Train : 133 obs × 6 features
Test  : 34 obs × 6 features


## Baseline

### Cross-validation baseline

In [4]:
baseline_models = {
    "ExtraTrees" : ExtraTreesRegressor(random_state=SEED, n_jobs=-1),
    "RandomForest" : RandomForestRegressor(random_state=SEED, n_jobs=-1),
    "XGBoost" : xgb.XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0),
}

print(f"\n{'Model':<20} {'R²':>8} {'RMSE':>8} {'MAE':>8}")
print("-" * 48)

baseline_results = {}
for name, mdl in baseline_models.items():
    preds = np.maximum(cross_val_predict(mdl, X_train, y_train, cv=cv), 0)
    r2 = r2_score(y_train, preds)
    rmse = np.sqrt(mean_squared_error(y_train, preds))
    mae = mean_absolute_error(y_train, preds)
    baseline_results[name] = {"r2": r2, "rmse": rmse, "mae": mae}
    print(f"{name:<20} {r2:>8.4f} {rmse:>8.2f} {mae:>8.2f}")



Model                      R²     RMSE      MAE
------------------------------------------------
ExtraTrees             0.5462     9.94     6.35
RandomForest           0.5460     9.94     6.36
XGBoost                0.5225    10.20     6.37


## Optuna Hyperparameter Tuning

### ExtraTrees

In [5]:
def objective_et(trial, metric="r2"):
    params = {
        'n_estimators' : trial.suggest_int('n_estimators', 100, 1000),
        'max_depth' : trial.suggest_int('max_depth', 5, 50),
        'min_samples_split' : trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features' : trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap' : trial.suggest_categorical('bootstrap', [True, False]),
        'random_state' : SEED
    }
    scores = cross_val_score(
        ExtraTreesRegressor(**params, n_jobs=-1),
        X_train, y_train, cv=cv, scoring=metric, n_jobs=-1
    )
    return scores.mean()

study_et = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study_et.optimize(objective_et, n_trials=N_TRIALS, show_progress_bar=True)

print(f"Best params : {study_et.best_params}")
print(f"Best CV R² : {study_et.best_value:.4f}")

best_et = ExtraTreesRegressor(**study_et.best_params, random_state=SEED, n_jobs=-1)
best_et.fit(X_train, y_train)


Best trial: 34. Best value: 0.545135: 100%|██████████| 50/50 [00:24<00:00,  2.03it/s]


Best params : {'n_estimators': 725, 'max_depth': 7, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': None, 'bootstrap': False}
Best CV R² : 0.5451


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",725
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",7
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",6
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the nu

### RandomForest

In [6]:
def objective_rf(trial, metric="r2"):
    params = {
        'n_estimators' : trial.suggest_int('n_estimators', 100, 1000),
        'max_depth' : trial.suggest_int('max_depth', 5, 50),
        'min_samples_split' : trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf'  : trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features' : trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap' : trial.suggest_categorical('bootstrap', [True, False]),
        'random_state' : SEED
    }
    scores = cross_val_score(
        RandomForestRegressor(**params, n_jobs=-1),
        X_train, y_train, cv=cv, scoring=metric, n_jobs=-1
    )
    return scores.mean()

study_rf = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study_rf.optimize(objective_rf, n_trials=N_TRIALS, show_progress_bar=True)

print(f"Best params : {study_rf.best_params}")
print(f"Best CV R² : {study_rf.best_value:.4f}")

best_rf = RandomForestRegressor(**study_rf.best_params, random_state=SEED, n_jobs=-1)
best_rf.fit(X_train, y_train)


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 47. Best value: 0.551079: 100%|██████████| 50/50 [00:19<00:00,  2.61it/s]

Best params : {'n_estimators': 342, 'max_depth': 22, 'min_samples_split': 12, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': True}
Best CV R² : 0.5511


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",342
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",22
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",12
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamp

### XGBoost

In [7]:
def objective_xgb(trial, metric="r2"):
    params = {
        'max_depth' : trial.suggest_int('max_depth', 3, 10),
        'learning_rate' : trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'n_estimators' : trial.suggest_int('n_estimators', 100, 1000),
        'reg_lambda' : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'subsample' : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state' : SEED,
        'verbosity' : 0,
        'n_jobs' : -1
    }
    scores = cross_val_score(
        xgb.XGBRegressor(**params),
        X_train, y_train, cv=cv, scoring=metric, n_jobs=-1
    )
    return scores.mean()

study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f"Best params : {study_xgb.best_params}")
print(f"Best CV R² : {study_xgb.best_value:.4f}")

best_xgb = xgb.XGBRegressor(**study_xgb.best_params)
best_xgb.fit(X_train, y_train)


Best trial: 28. Best value: 0.526359: 100%|██████████| 50/50 [00:08<00:00,  5.60it/s]


Best params : {'max_depth': 8, 'learning_rate': 0.003828524379565737, 'n_estimators': 800, 'reg_lambda': 4.779033244434582, 'subsample': 0.6884704013436093, 'colsample_bytree': 0.7165650600224711}
Best CV R² : 0.5264


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7165650600224711
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import

## Model Selection & Evaluation

### Best Model Selection

In [ ]:
tuned_models = {
    "ExtraTrees" : best_et,
    "RandomForest" : best_rf,
    "XGBoost" : best_xgb,
}

# Map each model name to its Optuna study to retrieve the CV R²
study_map = {
    "ExtraTrees" : study_et,
    "RandomForest" : study_rf,
    "XGBoost" : study_xgb,
}

# Select best model on CV R²
best_model_name = max(study_map, key=lambda k: study_map[k].best_value)
best_model = tuned_models[best_model_name]

print("CV R² summary (Optuna best values):")
print(f"\n{'Model':<20} {'CV R²':>8}")
print("-" * 30)
for name, study in study_map.items():
    marker = " ← best" if name == best_model_name else ""
    print(f"{name:<20} {study.best_value:>8.4f}{marker}")

print(f"\nBest model selected : {best_model_name}")
print(f"  CV R² (Optuna)    : {study_map[best_model_name].best_value:.4f}")


CV R² summary (Optuna best values):

Model                   CV R²
------------------------------
ExtraTrees             0.5451
RandomForest           0.5511 ← best
XGBoost                0.5264

Best model selected : RandomForest
  CV R² (Optuna)    : 0.5511


In [9]:
# CV RMSE with the best model
if best_model_name == "ExtraTrees":
    study_et = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
    study_et.optimize(lambda trial: objective_et(trial, metric='neg_root_mean_squared_error'), n_trials=N_TRIALS, show_progress_bar=True)
    best_rmse = -study_et.best_value
elif best_model_name == "RandomForest":
    study_rf = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
    study_rf.optimize(lambda trial: objective_rf(trial, metric='neg_root_mean_squared_error'), n_trials=N_TRIALS, show_progress_bar=True)
    best_rmse = -study_rf.best_value
elif best_model_name == "XGBoost":
    study_xgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
    study_xgb.optimize(lambda trial: objective_xgb(trial, metric='neg_root_mean_squared_error'), n_trials=N_TRIALS, show_progress_bar=True)
    best_rmse = -study_xgb.best_value
print(f"Best CV RMSE : {best_rmse:.4f}")

Best trial: 46. Best value: -12.9397: 100%|██████████| 50/50 [00:25<00:00,  1.97it/s]

Best CV RMSE : 12.9397


In [10]:
# CV MAE with the best model
if best_model_name == "ExtraTrees":
    study_et = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
    study_et.optimize(lambda trial: objective_et(trial, metric='neg_mean_absolute_error'), n_trials=N_TRIALS, show_progress_bar=True)
    best_mae = -study_et.best_value
elif best_model_name == "RandomForest":
    study_rf = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
    study_rf.optimize(lambda trial: objective_rf(trial, metric='neg_mean_absolute_error'), n_trials=N_TRIALS, show_progress_bar=True)
    best_mae = -study_rf.best_value
elif best_model_name == "XGBoost":
    study_xgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
    study_xgb.optimize(lambda trial: objective_xgb(trial, metric='neg_mean_absolute_error'), n_trials=N_TRIALS, show_progress_bar=True)
    best_mae = -study_xgb.best_value
print(f"Best CV MAE : {best_mae:.4f}")

Best trial: 31. Best value: -8.50237: 100%|██████████| 50/50 [00:24<00:00,  2.04it/s]

Best CV MAE : 8.5024


### Evaluation on test set

In [11]:
def evaluate_on_test(name_arg, model, X_test_arg, y_test_arg):
    """Return R², RMSE and MAE for a fitted model on the test set."""
    preds_in = np.maximum(model.predict(X_test_arg), 0)
    return {
        "model" : name_arg,
        "r2"    : r2_score(y_test_arg, preds_in),
        "rmse"  : np.sqrt(mean_squared_error(y_test_arg, preds_in)),
        "mae"   : mean_absolute_error(y_test_arg, preds_in),
        "preds" : preds_in
    }

test_results = {}
for name, mdl in tuned_models.items():
    if name == best_model_name:
        res = evaluate_on_test(name, mdl, X_test, y_test)
        test_results[name] = res

best_res = test_results[best_model_name]
print(f"\nBest model : {best_model_name}")
print(f"   CV R²  (Optuna) = {study_map[best_model_name].best_value:.4f}")
print(f"   R²  test        = {best_res['r2']:.4f}")
print(f"   RMSE test       = {best_res['rmse']:.2f} t/ha")
print(f"   MAE  test       = {best_res['mae']:.2f} t/ha")

# Save results
with open(RESULTS_PATH, 'w', encoding="utf-8") as f:
    f.write(f"Best model : {best_model_name}\n")
    f.write(f"   CV R²  (Optuna) = {study_map[best_model_name].best_value:.4f}\n")
    f.write(f"   R²  test        = {best_res['r2']:.4f}\n")
    f.write(f"   RMSE test       = {best_res['rmse']:.2f} t/ha\n")
    f.write(f"   MAE  test       = {best_res['mae']:.2f} t/ha\n")



Best model : RandomForest
   CV R²  (Optuna) = 0.5511
   R²  test        = 0.5201
   RMSE test       = 11.06 t/ha
   MAE  test       = 7.42 t/ha


## Final Training & Saving

### Retrain on full data (train + test) & save all models

In [12]:
# Combine train and test for final retraining
full_data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
X_full = full_data[FEATURES]
y_full = full_data[TARGET]

saved_paths = {}

for name, mdl in tuned_models.items():
    mdl.fit(X_full, y_full)
    path = f"{MODEL_DIR}{name.lower()}_tuned_{SAVE_DATE}.pkl"
    joblib.dump(mdl, path)
    saved_paths[name] = path
    print(f"{name} saved to {path}")


ExtraTrees saved to models/extratrees_tuned_20260528.pkl
RandomForest saved to models/randomforest_tuned_20260528.pkl
XGBoost saved to models/xgboost_tuned_20260528.pkl


### Save best model

In [13]:
best_path = f"{MODEL_DIR}BEST_{best_model_name.lower()}_{SAVE_DATE}.pkl"
joblib.dump(best_model, best_path)

print(f"\nBest model saved to {best_path}")



Best model saved to models/BEST_randomforest_20260528.pkl
